In [1]:
import yfinance as yf
import pandas as pd
import numpy as np

In [2]:
def dataFatcher(ticker):
    ticker = yf.Ticker(ticker)
    balance_sheet = ticker.balance_sheet
    quarterly_balance_sheet = ticker.quarterly_balance_sheet
    return balance_sheet, quarterly_balance_sheet
def dataOrganise(df):
    df = df.copy()
    # transpose so years become rows
    df = df.T
    # rename balance sheet fields
    rename_map = {
        "Total Assets": "total_assets",
        "Total Debt": "total_debt",
        "Net Debt": "net_debt",
        "Long Term Debt And Capital Lease Obligation": "long_term_debt",
        "Stockholders Equity": "stockholders_equity",
        "Retained Earnings": "retained_earnings",
        "Cash And Cash Equivalents": "cash",
        "Receivables": "receivables",
        "Accounts Payable": "accounts_payable",
        "Net PPE": "net_ppe",
        "Goodwill": "goodwill",
        "Other Intangible Assets": "intangible_assets",
        "Invested Capital": "invested_capital",
        "Share Issued": "shares_issued",
        "Minority Interest": "minority_interest"
    }
    df = df.rename(columns=rename_map)
    columns_need = [
        "total_assets","total_debt","net_debt","long_term_debt",
        "stockholders_equity","retained_earnings","cash",
        "receivables","accounts_payable","net_ppe","goodwill",
        "intangible_assets","invested_capital","shares_issued",
        "minority_interest"
    ]
    df = df.reindex(columns=columns_need)
    df["year"] = df.index
    df = df.reset_index(drop=True)
    return df

def addMatadata(df,ticker,type,name):
    df = df.copy()
    df['ticker'] = ticker
    df['type'] = type
    df['name'] = name
    return df


In [3]:
def cleaning(df):
    df = df.dropna(how="all")
    threshold = len(df) * 0.6
    df = df.dropna(axis=1, thresh=threshold)
    df = df.fillna(df.median(numeric_only=True))
    return df


In [4]:
def createRiskFeatures(df):

    df = df.copy()

    df["debt_to_equity"] = df["total_debt"] / df["stockholders_equity"]
    df["debt_to_assets"] = df["total_debt"] / df["total_assets"]
    df["net_debt_ratio"] = df["net_debt"] / df["total_assets"]

    df["equity_ratio"] = df["stockholders_equity"] / df["total_assets"]

    df["cash_ratio"] = df["cash"] / df["total_debt"]

    df["intangibles_ratio"] = df["intangible_assets"] / df["total_assets"]

    df["goodwill_ratio"] = df["goodwill"] / df["total_assets"]

    df["receivable_ratio"] = df["receivables"] / df["total_assets"]

    return df

In [5]:
def createGrowthFeatures(df):

    df = df.copy()

    df["asset_growth"] = df["total_assets"].pct_change()
    df["debt_growth"] = df["total_debt"].pct_change()
    df["equity_growth"] = df["stockholders_equity"].pct_change()

    return df

In [6]:
df = pd.read_csv(r"C:\Users\Deep\OneDrive\Desktop\AEGIS-FIN-main\stack\backend\core\sme_companies_loan_analysis.csv")
finalDf = pd.DataFrame()
for i in df['NSE/BSE Ticker']:
    mainBalanceSheet,_ = dataFatcher(i+".BO")
    mainBalanceSheet = dataOrganise(mainBalanceSheet)
    mainBalanceSheet = addMatadata(mainBalanceSheet,i,"",i)
    mainBalanceSheet = createRiskFeatures(mainBalanceSheet)
    mainBalanceSheet = createGrowthFeatures(mainBalanceSheet)
    finalDf = pd.concat([finalDf, mainBalanceSheet], axis=0, ignore_index=True)

finalDf = cleaning(finalDf)
display(finalDf)

,total_assets,total_debt,net_debt,long_term_debt,stockholders_equity,retained_earnings,cash,accounts_payable,net_ppe,intangible_assets,...,ticker,type,name,debt_to_equity,debt_to_assets,equity_ratio,cash_ratio,intangibles_ratio,asset_growth,equity_growth
0,1.315035e+10,1.987800e+08,3.205510e+09,1.123800e+08,1.247972e+10,1.393960e+09,381810000.0,4.722000e+07,5.208800e+08,1.036040e+09,...,ZAGGLE,,ZAGGLE,0.015928,0.015116,0.949003,1.920767,0.078784,-0.109796,-0.123832
1,6.961360e+09,8.662700e+08,6.566200e+08,2.423000e+08,5.753820e+09,5.147700e+08,79400000.0,1.963000e+07,1.673200e+08,5.801900e+08,...,ZAGGLE,,ZAGGLE,0.150556,0.124440,0.826537,0.091657,0.083344,-0.470633,-0.538946
2,2.347590e+09,1.413480e+09,1.014840e+09,6.719400e+08,4.875100e+08,7.457000e+07,195890000.0,9.219000e+07,2.415900e+08,1.775800e+08,...,ZAGGLE,,ZAGGLE,2.899387,0.602098,0.207664,0.138587,0.075644,-0.662768,-0.915272
3,9.265300e+08,7.031100e+08,6.376100e+08,5.343100e+08,-3.558000e+07,-1.544400e+08,7110000.0,1.073500e+08,9.630000e+07,5.929000e+07,...,ZAGGLE,,ZAGGLE,-19.761383,0.758864,-0.038401,0.010112,0.063991,-0.605327,-1.072983
4,2.343228e+10,3.036054e+09,6.629500e+08,1.358190e+09,1.177424e+10,5.277486e+09,495470000.0,1.749752e+09,4.463360e+09,5.824700e+07,...,ZAGGLE,,ZAGGLE,0.307028,0.176249,0.560446,0.142084,0.002651,-0.109796,-0.123832
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1451,1.097561e+10,5.570350e+08,5.494770e+08,1.358190e+09,9.385649e+09,8.533195e+09,7558000.0,1.920920e+08,8.197050e+08,4.190000e+05,...,GMBREW,,GMBREW,0.059350,0.050752,0.855137,0.013568,0.000038,-0.109796,-0.123832
1452,9.174399e+09,0.000000e+00,3.205510e+09,1.358190e+09,8.223204e+09,7.416444e+09,10449000.0,2.101010e+08,7.606100e+08,1.026000e+06,...,GMBREW,,GMBREW,0.000000,0.000000,0.896321,inf,0.000112,-0.164110,-0.123853
1453,7.691663e+09,3.036054e+09,3.205510e+09,1.358190e+09,6.817656e+09,6.010896e+09,16061000.0,8.904600e+07,7.803980e+08,1.633000e+06,...,GMBREW,,GMBREW,0.307028,0.176249,0.886370,0.142084,0.000212,-0.161617,-0.170925
1454,6.680233e+09,3.036054e+09,3.205510e+09,1.358190e+09,5.910396e+09,5.103636e+09,15249000.0,2.822700e+07,7.912570e+08,2.500000e+04,...,GMBREW,,GMBREW,0.307028,0.176249,0.884759,0.142084,0.000004,-0.131497,-0.133075


In [8]:
finalDf.shape
finalDf.columns

Index(['total_assets', 'total_debt', 'net_debt', 'long_term_debt',
       'stockholders_equity', 'retained_earnings', 'cash', 'accounts_payable',
       'net_ppe', 'intangible_assets', 'invested_capital', 'shares_issued',
       'year', 'ticker', 'type', 'name', 'debt_to_equity', 'debt_to_assets',
       'equity_ratio', 'cash_ratio', 'intangibles_ratio', 'asset_growth',
       'equity_growth'],
      dtype='str')